# SMS & Email Spam Classifier
### End-to-End Machine Learning Pipeline: From Raw Text to Production Model

This notebook demonstrates how to build, train, evaluate, and interpret a machine learning system that classifies messages as **HAM** (legitimate) or **SPAM** (unsolicited / malicious).

#### Key Objectives:
1. **Clean & Parse Raw Data**: Ingest UCI SMS Spam Collection with `latin-1` encoding, drop redundant columns, remove duplicates, and map labels (`ham` &rarr; 0, `spam` &rarr; 1).
2. **Text Preprocessing**: Lowercasing, punctuation & number stripping, stopword elimination, and Porter stemming using **NLTK**.
3. **Exploratory Data Analysis (EDA)**: Analyze class imbalance and investigate message length and word count distributions.
4. **Feature Extraction**: Unigram and bigram **TF-IDF Vectorization** (with strict train-test separation to avoid data leakage).
5. **Hyperparameter Tuning**: Compare **Multinomial Naive Bayes**, **Logistic Regression**, and **Linear Support Vector Classifier (LinearSVC)** using 5-fold Stratified Cross-Validation.
6. **Evaluation**: Assess Accuracy, Precision, Recall, F1-Score, Confusion Matrices, ROC-AUC, and False Positive Rates.
7. **Interpretability**: Extract top 20 spam-predicting tokens and bigrams.
8. **Inference**: Test custom messages in real time with confidence scores.

## 1. Environment Setup & Imports

In [ ]:
import os
import re
import string
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report
)

# Set plots style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
print("Libraries loaded successfully!")

## 2. Ingest and Clean the Raw Dataset
- Read with `encoding='latin-1'`.
- Drop empty unnamed columns (`Unnamed: 2`, `3`, `4`).
- Rename `v1` &rarr; `label` and `v2` &rarr; `text`.
- Map `ham` &rarr; 0 and `spam` &rarr; 1.
- Drop duplicate rows.

In [ ]:
DATA_PATH = os.path.join("data", "spam.csv")
if not os.path.exists(DATA_PATH):
    DATA_PATH = "spam.csv"

df_raw = pd.read_csv(DATA_PATH, encoding="latin-1")
print(f"Raw dataset shape: {df_raw.shape}")

# Drop unneeded columns
unnamed_cols = [c for c in df_raw.columns if "Unnamed" in c]
df = df_raw.drop(columns=unnamed_cols)

# Rename columns
df = df.rename(columns={"v1": "label", "v2": "text"})

# Encode labels: ham -> 0, spam -> 1
df["label"] = df["label"].map({"ham": 0, "spam": 1})

# Remove duplicates
before_len = len(df)
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
after_len = len(df)
print(f"Dropped {before_len - after_len} duplicates. Cleaned dataset has {after_len} unique messages.")
df.head()

## 3. Exploratory Data Analysis (EDA)
Let us examine:
1. Class imbalance ratio between Ham and Spam.
2. Character length and word count distributions.

In [ ]:
# Compute message character length and word counts
df["char_length"] = df["text"].apply(len)
df["word_count"] = df["text"].apply(lambda s: len(s.split()))

# 1. Class Distribution
counts = df["label"].value_counts().rename({0: "Ham (0)", 1: "Spam (1)"})
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(counts.index, counts.values, color=["#2b6cb0", "#e53e3e"], width=0.45)
for bar, c in zip(bars, counts.values):
    pct = (c / len(df)) * 100
    ax.text(bar.get_x() + bar.get_width()/2, c + 50, f"{c:,} ({pct:.1f}%)", ha="center", fontweight="bold")
ax.set_title("Class Imbalance in Dataset", fontsize=12, fontweight="bold")
ax.set_ylabel("Count")
plt.show()

# 2. Message Length Distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[df["label"] == 0]["char_length"].clip(upper=350), ax=ax1, color="#2b6cb0", label="Ham", kde=True)
sns.histplot(df[df["label"] == 1]["char_length"].clip(upper=350), ax=ax1, color="#e53e3e", label="Spam", kde=True)
ax1.set_title("Character Length")
ax1.legend()

sns.histplot(df[df["label"] == 0]["word_count"].clip(upper=60), ax=ax2, color="#2b6cb0", label="Ham", kde=True)
sns.histplot(df[df["label"] == 1]["word_count"].clip(upper=60), ax=ax2, color="#e53e3e", label="Spam", kde=True)
ax2.set_title("Word Count")
ax2.legend()
plt.tight_layout()
plt.show()

## 4. Text Preprocessing with NLTK
We apply a robust NLP preprocessing pipeline:
1. **Lowercasing**: Uniform word representations.
2. **Regex Cleaning**: Stripping URLs, emails, and standalone numbers.
3. **Punctuation Removal**: Filtering special symbols.
4. **Tokenization**: Breaking text into individual word tokens.
5. **Stopword Elimination**: Removing non-informative words (`the`, `is`, `at`, etc.).
6. **Stemming**: Normalizing words to their morphological roots with `PorterStemmer`.

In [ ]:
from src.preprocess import clean_text

# Test on sample
sample_raw = "WINNER!! As a valued network customer you have won a å£900 prize reward! Claim code KL341."
print("Sample Raw :", sample_raw)
print("Cleaned    :", clean_text(sample_raw))

# Apply to entire dataset
df["cleaned_text"] = df["text"].apply(clean_text)
df[["text", "cleaned_text", "label"]].head()

## 5. Stratified Train/Test Split & TF-IDF Vectorization
- 80% Train, 20% Test stratified by label to maintain the 87.4% / 12.6% class ratio.
- **TfidfVectorizer**: Unigrams + Bigrams (`ngram_range=(1, 2)`), `max_features=5000`.
- **Critical Rule**: Vectorizer is fitted **only on the training set** to prevent data leakage.

In [ ]:
X = df["cleaned_text"]
y = df["label"]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Train samples: {len(X_train_raw)} | Test samples: {len(X_test_raw)}")

# Fit vectorizer on TRAIN only
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True, min_df=2)
X_train_vec = vectorizer.fit_transform(X_train_raw)
X_test_vec = vectorizer.transform(X_test_raw)

print(f"Feature matrix train shape: {X_train_vec.shape}")
print(f"Feature matrix test shape : {X_test_vec.shape}")

## 6. Model Training & 5-Fold Stratified GridSearchCV
We train and tune three distinct model architectures:
1. **Multinomial Naive Bayes (`MultinomialNB`)**: Classic probabilistic text baseline.
2. **Logistic Regression**: Linear probabilistic classifier with `class_weight='balanced'`.
3. **Linear Support Vector Classifier (`LinearSVC`)**: Maximum margin classifier with `class_weight='balanced'`.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_config = {
    "Multinomial Naive Bayes": (
        MultinomialNB(),
        {"alpha": [0.01, 0.05, 0.1, 0.5, 1.0, 2.0]}
    ),
    "Logistic Regression": (
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
        {"C": [0.1, 0.5, 1.0, 5.0, 10.0], "solver": ["liblinear", "lbfgs"]}
    ),
    "Linear SVM": (
        LinearSVC(max_iter=3000, class_weight="balanced", random_state=42),
        {"C": [0.01, 0.05, 0.1, 0.5, 1.0, 2.0]}
    )
}

trained_models = {}
for name, (estimator, params) in models_config.items():
    print(f"Tuning {name}...")
    grid = GridSearchCV(estimator, params, cv=cv, scoring="f1", n_jobs=1)
    grid.fit(X_train_vec, y_train)
    trained_models[name] = grid.best_estimator_
    print(f"  Best Parameters: {grid.best_params_} (CV F1: {grid.best_score_:.4f})")

## 7. Model Evaluation & Comparison on Test Set
Spam filtering demands **high precision on spam** (low false-positive rate), because misclassifying a legitimate email/SMS as spam can cause users to miss critical messages.

In [ ]:
results = []
cm_dict = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test_vec)
    if hasattr(model, "predict_proba"):
        y_scores = model.predict_proba(X_test_vec)[:, 1]
    else:
        y_scores = model.decision_function(X_test_vec)
    
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    cm_dict[name] = cm
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (Spam)": precision_score(y_test, y_pred, pos_label=1),
        "Recall (Spam)": recall_score(y_test, y_pred, pos_label=1),
        "F1-Score (Spam)": f1_score(y_test, y_pred, pos_label=1),
        "ROC-AUC": roc_auc_score(y_test, y_scores),
        "False Positives": fp,
        "FPR (%)": (fp / (fp + tn)) * 100
    })

results_df = pd.DataFrame(results)
display(results_df.style.highlight_max(subset=["Accuracy", "Precision (Spam)", "F1-Score (Spam)", "ROC-AUC"], color="#c6f6d5"))

## 8. Confusion Matrices & Key Spam Keywords
Let us inspect the confusion matrix side-by-side and visualize the top 20 words that most strongly indicate spam according to Logistic Regression.

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, cm) in zip(axes, cm_dict.items()):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Ham", "Spam"], yticklabels=["Ham", "Spam"])
    ax.set_title(f"{name}\nConfusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

# Top 20 Spam Words (Logistic Regression)
lr_model = trained_models["Logistic Regression"]
feature_names = np.array(vectorizer.get_feature_names_out())
top_indices = np.argsort(lr_model.coef_[0])[::-1][:20]

plt.figure(figsize=(8, 6))
plt.barh(feature_names[top_indices][::-1], lr_model.coef_[0][top_indices][::-1], color="#e53e3e")
plt.title("Top 20 Spam-Indicating Words & Bigrams (Logistic Regression)", fontweight="bold")
plt.xlabel("Coefficient Weight")
plt.show()

## 9. Real-Time Inference Demo
Let's test the production predictor on custom email/SMS texts.

In [ ]:
from src.predict import SpamPredictor

predictor = SpamPredictor()

test_samples = [
    "Hey, are we still meeting for coffee this afternoon?",
    "URGENT! You have won a guaranteed å£1000 prize or holiday. Call 09061701461 now!",
    "Can you review the attached project documentation before tomorrow's call?",
    "Claim your free bonus spins now! Visit bit.ly/free-coins or text STOP to 80082"
]

for text in test_samples:
    out = predictor.predict_one(text)
    print(f"Message    : {text}")
    print(f"Prediction : [{out['label']}] (Confidence: {out['confidence']*100:.2f}% | Spam Prob: {out['spam_probability']*100:.2f}%)")
    print("-" * 70)